# 10. 관계형 Critical Fact v2 평가

이 노트북은 기존 파일과 과거 OCR 산출물을 수정하지 않고 다음 작업을 오프라인으로 수행한다.

1. `critical_rules_v1.json`의 복합 fact를 원자 값으로 분해한다.
2. 각 값에 혜택/수수료 ID, 대상, 조건, 정규 단위, 원본 페이지와 JSON 경로를 연결한다.
3. 기존 09의 Upstage 1회, API Luna 2회, API Terra 2회 구조화 예측을 동일한 v2 기준으로 재평가한다.
4. 원자 관계 정확도, 수치 정확도, 관계 그룹 정확도, 위험 오답률과 오류 유형을 기록한다.

주의: 09의 예측 스키마에 없던 보충 라벨은 v2에는 포함되지만 이번 오프라인 재평가의 점수 분모에서는 제외한다.
이는 정답 누락을 숨기지 않으면서도, 모델이 애초에 요청받지 않은 필드를 오답으로 처리하지 않기 위한 구분이다.


In [1]:
from __future__ import annotations

import csv
import json
import os
import re
import statistics
import unicodedata
from collections import Counter, defaultdict
from datetime import datetime, timezone
from pathlib import Path
from typing import Any, Iterable


def find_project_root() -> Path:
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "data/ocr_benchmark/gold/critical_rules/critical_rules_v1.json").exists():
            return candidate
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")


PROJECT_ROOT = find_project_root()
V1_PATH = PROJECT_ROOT / "data/ocr_benchmark/gold/critical_rules/critical_rules_v1.json"
V2_PATH = PROJECT_ROOT / "data/ocr_benchmark/gold/critical_rules/critical_rules_v2.json"
STRUCTURED_ROOT = PROJECT_ROOT / "data/ocr_benchmark/gold/structured"
OUTPUT_ROOT = PROJECT_ROOT / "notebooks/data/10_relational_critical_fact_evaluation"
RUN_ID = os.getenv("RELATIONAL_EVAL_RUN_ID") or datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
RUN_ROOT = OUTPUT_ROOT / "runs" / RUN_ID

SOURCE_SPECS = [
    {
        "run_name": "upstage_baseline",
        "model_group": "Upstage Document Parse",
        "prediction_root": PROJECT_ROOT / "notebooks/data/09_core_numeric_condition_ocr_evaluation/runs/20260807T144000Z_upstage_baseline/predictions/baseline_upstage_document_parse",
    },
    {
        "run_name": "luna_original_repeat_1",
        "model_group": "OpenAI API Luna original",
        "prediction_root": PROJECT_ROOT / "notebooks/data/09_core_numeric_condition_ocr_evaluation/runs/20260807T143000Z_repeat01/predictions/api_luna_original",
    },
    {
        "run_name": "luna_original_repeat_2",
        "model_group": "OpenAI API Luna original",
        "prediction_root": PROJECT_ROOT / "notebooks/data/09_core_numeric_condition_ocr_evaluation/runs/20260807T144500Z_api_repeat01/predictions/api_luna_original",
    },
    {
        "run_name": "terra_original_repeat_1",
        "model_group": "OpenAI API Terra original",
        "prediction_root": PROJECT_ROOT / "notebooks/data/09_core_numeric_condition_ocr_evaluation/runs/20260807T143000Z_repeat01/predictions/api_terra_original",
    },
    {
        "run_name": "terra_original_repeat_2",
        "model_group": "OpenAI API Terra original",
        "prediction_root": PROJECT_ROOT / "notebooks/data/09_core_numeric_condition_ocr_evaluation/runs/20260807T144500Z_api_repeat01/predictions/api_terra_original",
    },
]

# v1에서 critical=false였지만 금액·수수료·혜택 조건상 안전성 평가에 포함해야 하는 라벨.
SUPPLEMENTARY_NUMERIC_IDS = {
    "BC": {"point_expiry", "lounge_hours", "lounge_quota", "lounge_period"},
    "NH": {"international_brand_fee_rate", "overseas_service_fee_rate"},
    "ibk": {"international_brand_visa_fee_rate", "international_brand_mastercard_fee_rate", "overseas_service_fee_rate"},
    "lotte": {"master_international_brand_fee_rate", "amex_international_brand_fee_rate", "credit_card_overseas_service_fee_rate"},
    "samsung": {"international_mastercard_fee_rate", "international_visa_fee_rate", "overseas_service_fee_rate"},
    "shinhan": {"international_brand_fee_rate", "credit_card_overseas_service_fee_rate", "check_card_overseas_service_fee_rate"},
    "woori": {"overseas_atm_withdrawal_fee", "lounge_voucher_validity"},
}

UNORDERED_LIST_KEYS = {
    "targets", "target_merchants", "target_areas", "categories", "payment_types",
    "brands", "all_day_targets", "night_targets", "merchant_scope", "scope",
    "ranking_scope", "eligible_member",
}

# semantic field에 없는 별도 안전 규칙으로서 관계 점수에도 포함할 numeric label.
RELATION_PRIMARY_NUMERIC_IDS = {
    "ibk": {"fuel_spend_monthly_recognition_limit"},
}


def read_json(path: Path) -> dict[str, Any]:
    return json.loads(path.read_text(encoding="utf-8"))


def write_json(path: Path, value: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")


def write_json_without_overwrite(path: Path, value: Any) -> None:
    serialized = json.dumps(value, ensure_ascii=False, indent=2) + "\n"
    if path.exists():
        if path.read_text(encoding="utf-8") != serialized:
            raise FileExistsError(f"기존 파일과 내용이 달라 덮어쓰지 않습니다: {path}")
        return
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(serialized, encoding="utf-8")


def write_csv(path: Path, rows: list[dict[str, Any]]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fieldnames = list(rows[0]) if rows else []
    with path.open("w", encoding="utf-8", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        if fieldnames:
            writer.writeheader()
            writer.writerows(rows)


def flatten(value: Any, path: tuple[Any, ...] = ()) -> Iterable[tuple[tuple[Any, ...], Any]]:
    if isinstance(value, dict):
        if not value:
            yield path, value
        for key, item in value.items():
            yield from flatten(item, path + (key,))
    elif isinstance(value, list):
        if not value:
            yield path, value
        for index, item in enumerate(value):
            yield from flatten(item, path + (index,))
    else:
        yield path, value


def path_text(path: tuple[Any, ...]) -> str:
    result = "$"
    for token in path:
        result += f"[{token}]" if isinstance(token, int) else f".{token}"
    return result


def safe_id(text: str) -> str:
    normalized = re.sub(r"[^0-9A-Za-z가-힣]+", "_", text).strip("_")
    return normalized or "root"


def canonical_unit(unit: Any, metric: str, value: Any) -> str:
    aliases = {
        "won": "KRW", "원": "KRW", "KRW": "KRW", "ratio": "RATIO", "USD": "USD",
        "point": "POINT", "M_point": "M_POINT", "mile": "MILE",
        "%": "RATIO", "rate": "RATIO", "RATE": "RATIO", "포인트": "POINT",
        "M포인트": "M_POINT", "마일": "MILE", "miles": "MILE",
        "KRW/L": "KRW_PER_LITER", "times/year": "COUNT_PER_YEAR",
        "KRW_per_liter": "KRW_PER_LITER", "per_year": "COUNT_PER_YEAR",
        "per_month": "COUNT_PER_MONTH", "hour": "HOUR", "year": "YEAR",
        "month": "MONTH", "business_day": "BUSINESS_DAY",
    }
    if unit in aliases:
        return aliases[unit]
    key = metric.casefold()
    if "krw_per_liter" in key or ("liter" in key and "discount" in key):
        return "KRW_PER_LITER"
    if key == "won" or key.endswith(".won") or "krw" in key or "annual_fee" in key or key.endswith("_fee"):
        return "KRW"
    if "point_expiry" in key and key.endswith(".year"):
        return "YEAR"
    if "point_expiry" in key and key.endswith(".month"):
        return "MONTH"
    if "lounge_quota" in key and key.endswith(".year"):
        return "COUNT_PER_YEAR"
    if "lounge_quota" in key and key.endswith(".day"):
        return "COUNT_PER_DAY"
    if "rate" in key:
        return "RATIO"
    if "m_point" in key:
        return "M_POINT"
    if "point" in key:
        return "POINT"
    if "mile" in key:
        return "MILE"
    if key in {"annual_limit", "per_year"}:
        return "COUNT_PER_YEAR"
    if key in {"monthly_limit", "per_month"}:
        return "COUNT_PER_MONTH"
    if key == "per_day":
        return "COUNT_PER_DAY"
    if isinstance(value, bool):
        return "BOOLEAN"
    if value is None:
        return "NONE"
    if isinstance(value, str):
        return "TEXT"
    return "COUNT" if isinstance(value, int) else "NUMBER"


def normalize_unit(unit: Any, expected_unit: str | None = None) -> Any:
    if unit is None:
        return None
    if str(unit).casefold() in {"회", "time", "times", "count"} and expected_unit and expected_unit.startswith("COUNT_PER_"):
        # 월/연 구분은 고정 relation metric(예: monthly_limit, annual_limit)에 이미 있다.
        return expected_unit
    if unit == "KRW" and expected_unit == "KRW_PER_LITER":
        return expected_unit
    if isinstance(unit, str) and unit.startswith("원당") and expected_unit == "KRW":
        return expected_unit
    return canonical_unit(unit, "", 0)


def normalize_text(value: str) -> str:
    value = unicodedata.normalize("NFKC", value).casefold()
    return re.sub(r"\s+", "", value)


def values_equal(expected: Any, actual: Any) -> bool:
    if isinstance(expected, bool) or isinstance(actual, bool):
        return type(expected) is type(actual) and expected == actual
    if isinstance(expected, (int, float)) and isinstance(actual, (int, float)):
        return abs(float(expected) - float(actual)) < 1e-12
    if isinstance(expected, str) and isinstance(actual, str):
        return normalize_text(expected) == normalize_text(actual)
    return expected == actual


def value_at(value: Any, path: tuple[Any, ...]) -> tuple[bool, Any]:
    current = value
    for token in path:
        if isinstance(token, int):
            if not isinstance(current, list) or token >= len(current):
                return False, None
            current = current[token]
        else:
            if not isinstance(current, dict) or token not in current:
                return False, None
            current = current[token]
    return True, current


def scalar_values(value: Any) -> list[Any]:
    if isinstance(value, dict):
        return [leaf for item in value.values() for leaf in scalar_values(item)]
    if isinstance(value, list):
        return [leaf for item in value for leaf in scalar_values(item)]
    return [value]


def infer_targets(root: Any, field_id: str) -> list[Any]:
    target_keys = {
        "target", "targets", "target_merchants", "target_areas", "categories",
        "merchant_scope", "scope", "ranking_scope", "payment_destination",
        "eligible_member", "all_day_targets", "night_targets",
    }
    found: list[Any] = []

    def visit(value: Any) -> None:
        if isinstance(value, dict):
            for key, item in value.items():
                if key in target_keys:
                    found.extend(scalar_values(item))
                else:
                    visit(item)
        elif isinstance(value, list):
            for item in value:
                visit(item)

    visit(root)
    if not found and "exclusions" in field_id:
        found.append("제외 대상")
    unique: list[Any] = []
    for item in found:
        if item not in unique:
            unique.append(item)
    return unique


def parse_spend_condition(text: str) -> list[dict[str, Any]]:
    compact = normalize_text(text).replace(",", "")
    range_match = re.fullmatch(r"(\d+)~(\d+)만원", compact)
    if range_match:
        low, high = (int(value) * 10000 for value in range_match.groups())
        return [
            {"field": "previous_month_spend", "operator": ">=", "value": low, "unit": "KRW"},
            {"field": "previous_month_spend", "operator": "<", "value": high, "unit": "KRW"},
        ]
    minimum_match = re.fullmatch(r"(\d+)만원이상", compact)
    if minimum_match:
        return [{"field": "previous_month_spend", "operator": ">=", "value": int(minimum_match.group(1)) * 10000, "unit": "KRW"}]
    return []


def condition_object(key: str, value: Any) -> list[dict[str, Any]]:
    if key == "previous_month_spend" and isinstance(value, str):
        parsed = parse_spend_condition(value)
        if parsed:
            return parsed
    if key.endswith("minimum_krw") or key.endswith("spend_minimum_krw"):
        return [{"field": key.removesuffix("_minimum_krw"), "operator": ">=", "value": value, "unit": "KRW"}]
    if key in {"previous_month_spend_required", "previous_month_spend_requirement", "payment_requirement", "night_hours", "selection_rule"}:
        return [{"field": key, "operator": "=", "value": value, "unit": canonical_unit(None, key, value)}]
    return []


def infer_conditions(root: Any, path: tuple[Any, ...]) -> list[dict[str, Any]]:
    condition_key_pattern = re.compile(r"previous_month|minimum|requirement|required|night_hours|selection_rule")
    conditions: list[dict[str, Any]] = []
    current = root
    for depth, token in enumerate(path):
        if isinstance(current, dict):
            for key, value in current.items():
                if key == token or not condition_key_pattern.search(key):
                    continue
                if not isinstance(value, (dict, list)):
                    conditions.extend(condition_object(key, value) or [{"field": key, "operator": "=", "value": value, "unit": canonical_unit(None, key, value)}])
        ok, current = value_at(root, path[: depth + 1])
        if not ok:
            break
    unique: list[dict[str, Any]] = []
    for item in conditions:
        if item not in unique:
            unique.append(item)
    return unique


def relation_group_path(path: tuple[Any, ...]) -> tuple[Any, ...]:
    for index in range(len(path) - 1, -1, -1):
        if isinstance(path[index], int):
            return path[: index + 1]
    if len(path) >= 2:
        return path[:-1]
    return ()


def is_unordered_member(field_id: str, path: tuple[Any, ...]) -> bool:
    if not path or not isinstance(path[-1], int):
        return False
    parent_key = path[-2] if len(path) >= 2 else None
    return (isinstance(parent_key, str) and parent_key in UNORDERED_LIST_KEYS) or "exclusions" in field_id


def risk_component(metric: str, field_id: str, value: Any) -> str:
    key = f"{field_id}.{metric}".casefold()
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return "numeric_value"
    if any(token in key for token in ("target", "categories", "scope", "merchant", "brand")):
        return "target"
    if any(token in key for token in ("previous_month", "minimum", "requirement", "hours", "selection_rule")):
        return "condition"
    if "exclusion" in key:
        return "exclusion"
    return "descriptive_value"


v1 = read_json(V1_PATH)
structured_by_card: dict[tuple[str, str], dict[str, Any]] = {}
for path in sorted(STRUCTURED_ROOT.glob("*/*.json")):
    data = read_json(path)
    structured_by_card[(data["issuer"], data["card_name"])] = data

In [2]:
def make_atomic_fact(
    *, card: dict[str, Any], v1_fact: dict[str, Any], root_expected: Any,
    leaf_path: tuple[Any, ...], expected_value: Any, stated_unit: Any = None,
    prediction_path: tuple[Any, ...] | None, match_mode: str = "scalar",
    label_origin: str = "v1", evaluation_role: str = "relation_primary",
    prediction_supported: bool = True,
    surface_text: str | None = None, audit_reason: str | None = None,
) -> dict[str, Any]:
    if leaf_path and isinstance(leaf_path[-1], str):
        metric = leaf_path[-1]
    elif len(leaf_path) >= 2 and isinstance(leaf_path[-2], str):
        metric = leaf_path[-2]
    else:
        metric = v1_fact["field_id"]
    unit = canonical_unit(stated_unit, f"{v1_fact['field_id']}.{metric}", expected_value)
    group_path = relation_group_path(leaf_path)
    relation_type = "table_row" if any(isinstance(token, int) for token in group_path) else ("object_row" if group_path else "benefit")
    suffix = safe_id(path_text(leaf_path))
    return {
        "fact_id": f"v2__{v1_fact['fact_id']}__{suffix}",
        "benefit_or_fee_id": v1_fact["field_id"],
        "target": infer_targets(root_expected, v1_fact["field_id"]),
        "metric": metric,
        "expected": {"value": expected_value, "unit": unit},
        "conditions": infer_conditions(root_expected, leaf_path),
        "critical": True,
        "risk_component": risk_component(metric, v1_fact["field_id"], expected_value),
        "relation": {
            "attribute_path": path_text(leaf_path),
            "group_path": path_text(group_path),
            "group_type": relation_type,
            "match_mode": match_mode,
        },
        "source": {
            "page_num": v1_fact.get("page_num"),
            "structured_source_file": card["source_file"],
            "v1_fact_id": v1_fact.get("fact_id"),
            "v1_fact_kind": v1_fact.get("fact_kind"),
            "context_terms": v1_fact.get("context_terms", []),
            "surface_text": surface_text if surface_text is not None else v1_fact.get("source", {}).get("surface_text"),
            "prediction_path": list(prediction_path) if prediction_path is not None else None,
            "prediction_supported": prediction_supported,
            "label_origin": label_origin,
            "evaluation_role": evaluation_role,
            "audit_reason": audit_reason,
        },
    }


def semantic_atomic_facts(card: dict[str, Any], fact: dict[str, Any]) -> list[dict[str, Any]]:
    root = fact["expected"]
    rows = []
    for leaf_path, value in flatten(root):
        match_mode = "membership" if is_unordered_member(fact["field_id"], leaf_path) else "scalar"
        prediction_path = leaf_path[:-1] if match_mode == "membership" else leaf_path
        rows.append(make_atomic_fact(
            card=card, v1_fact=fact, root_expected=root, leaf_path=leaf_path,
            expected_value=value, prediction_path=prediction_path, match_mode=match_mode,
        ))
    return rows


def numeric_atomic_facts(
    card: dict[str, Any], fact: dict[str, Any], *, label_origin: str = "v1",
    evaluation_role: str = "numeric_probe", prediction_supported: bool = True,
    audit_reason: str | None = None,
) -> list[dict[str, Any]]:
    expected = fact["expected"]
    root_value = expected.get("value")
    stated_unit = expected.get("unit")
    rows = []
    for leaf_path, value in flatten(root_value):
        prediction_path = ("value",) + leaf_path
        rows.append(make_atomic_fact(
            card=card, v1_fact=fact, root_expected=root_value,
            leaf_path=leaf_path or ("value",), expected_value=value,
            stated_unit=stated_unit, prediction_path=prediction_path,
            label_origin=label_origin, evaluation_role=evaluation_role,
            prediction_supported=prediction_supported,
            surface_text=fact.get("source", {}).get("surface_text"), audit_reason=audit_reason,
        ))
        if stated_unit is not None:
            rows[-1]["source"]["unit_prediction_path"] = ["unit"] if prediction_supported else None
    return rows


v2_cards = []
audit_rows = []
for card in v1["cards"]:
    semantic_rows: list[dict[str, Any]] = []
    numeric_probes: list[dict[str, Any]] = []
    has_semantic = any(fact["fact_kind"] == "semantic_field" for fact in card["facts"])
    for fact in card["facts"]:
        if fact["fact_kind"] == "semantic_field":
            semantic_rows.extend(semantic_atomic_facts(card, fact))
        else:
            role = "relation_primary" if not has_semantic or fact["field_id"] in RELATION_PRIMARY_NUMERIC_IDS.get(card["issuer"], set()) else "numeric_probe"
            numeric_probes.extend(numeric_atomic_facts(card, fact, evaluation_role=role))

    structured = structured_by_card[(card["issuer"], card["card_name"])]
    supplementary = []
    allowed_ids = SUPPLEMENTARY_NUMERIC_IDS.get(card["issuer"], set())
    for item in structured.get("numeric_labels", []):
        if item["id"] not in allowed_ids:
            continue
        pseudo = {
            "fact_id": f"supplementary_numeric__{item['id']}",
            "fact_kind": "numeric_fact",
            "field_id": item["id"],
            "page_num": item.get("page_num"),
            "context_terms": item.get("context_terms", []),
            "expected": {"value": item.get("normalized_value"), "unit": item.get("unit")},
            "source": {"surface_text": item.get("surface_text"), "structured_numeric_id": item["id"]},
        }
        supplementary.extend(numeric_atomic_facts(
            card, pseudo, label_origin="safety_audit_addition", evaluation_role="safety_audit_pending",
            prediction_supported=False,
            audit_reason="v1의 추출 스키마에는 없었으나 혜택·수수료·이용 조건 정확성에 필요한 항목",
        ))

    facts = semantic_rows + numeric_probes + supplementary
    ids = [item["fact_id"] for item in facts]
    if len(ids) != len(set(ids)):
        duplicates = [key for key, count in Counter(ids).items() if count > 1]
        raise ValueError(f"중복 fact_id: {card['issuer']}/{card['card_name']} {duplicates}")
    v2_cards.append({
        "issuer": card["issuer"],
        "card_name": card["card_name"],
        "source_file": card["source_file"],
        "facts": facts,
    })
    audit_rows.append({
        "issuer": card["issuer"], "card_name": card["card_name"],
        "semantic_atomic_facts": len(semantic_rows),
        "v1_numeric_probe_facts": sum(item["source"]["evaluation_role"] == "numeric_probe" for item in numeric_probes),
        "v1_numeric_relation_facts": sum(item["source"]["evaluation_role"] == "relation_primary" for item in numeric_probes),
        "supplementary_safety_facts": len(supplementary),
        "total_v2_facts": len(facts),
        "prediction_supported_facts": sum(item["source"]["prediction_supported"] for item in facts),
    })

v2 = {
    "schema_version": "relational_critical_fact_gold_v2",
    "derived_from": str(V1_PATH.relative_to(PROJECT_ROOT)),
    "annotation_scope": (
        "v1의 critical semantic/numeric labels를 모두 원자 값으로 보존하고 관계 경로를 연결했다. "
        "semantic 원자는 관계 점수, numeric 원자는 수치 probe 점수로 역할을 분리해 중복값의 의미 라벨을 삭제하지 않는다. "
        "혜택·수수료·서비스 조건상 중요한 일부 non-critical numeric label을 별도 안전성 감사 항목으로 추가했다. "
        "추가 항목은 09 예측 스키마 밖이므로 이번 오프라인 점수 분모에서는 제외한다."
    ),
    "cards": v2_cards,
}

write_json_without_overwrite(V2_PATH, v2)
write_csv(RUN_ROOT / "label_audit.csv", audit_rows)
print(f"v2 생성/확인: {len(v2_cards)}개 카드, {sum(len(card['facts']) for card in v2_cards)}개 원자 fact")
print(f"보충 안전성 fact: {sum(row['supplementary_safety_facts'] for row in audit_rows)}개")

v2 생성/확인: 10개 카드, 465개 원자 fact
보충 안전성 fact: 22개


In [3]:
def prediction_file(spec: dict[str, Any], card: dict[str, Any]) -> Path:
    return spec["prediction_root"] / card["issuer"] / f"{card['card_name']}.json"


missing_prediction_files = [
    str(prediction_file(spec, card).relative_to(PROJECT_ROOT))
    for spec in SOURCE_SPECS for card in v2_cards
    if not prediction_file(spec, card).exists()
]
if missing_prediction_files:
    raise FileNotFoundError("필요한 기존 예측 파일이 없습니다:\n" + "\n".join(missing_prediction_files))


def get_prediction_value(predictions: dict[str, Any], fact: dict[str, Any]) -> tuple[str, Any, Any]:
    source = fact["source"]
    if not source["prediction_supported"]:
        return "unsupported", None, None
    v1_fact_id = source["v1_fact_id"]
    if v1_fact_id not in predictions:
        return "missing", None, None
    root = predictions[v1_fact_id]
    path = tuple(source["prediction_path"] or [])
    found, actual = value_at(root, path)
    if not found:
        return "missing", None, None
    unit_actual = None
    unit_path = source.get("unit_prediction_path")
    if unit_path:
        _, unit_actual = value_at(root, tuple(unit_path))
    return "available", actual, unit_actual


def classify_error(fact: dict[str, Any], presence: str, value_exact: bool, unit_exact: bool | None) -> str:
    if presence == "unsupported":
        return "unsupported_by_v1_prediction_schema"
    if presence == "missing":
        return "missing_prediction"
    if not value_exact:
        component = fact["risk_component"]
        return {
            "numeric_value": "numeric_value_mismatch",
            "target": "target_mismatch",
            "condition": "condition_mismatch",
            "exclusion": "exclusion_mismatch",
        }.get(component, "descriptive_value_mismatch")
    if unit_exact is False:
        return "unit_mismatch"
    return "matched"


details: list[dict[str, Any]] = []
for spec in SOURCE_SPECS:
    for card in v2_cards:
        payload = read_json(prediction_file(spec, card))
        predictions = payload.get("predictions", payload)
        for fact in card["facts"]:
            presence, actual, actual_unit = get_prediction_value(predictions, fact)
            expected = fact["expected"]["value"]
            if presence == "available" and fact["relation"]["match_mode"] == "membership":
                value_exact = isinstance(actual, list) and any(values_equal(expected, item) for item in actual)
            else:
                value_exact = presence == "available" and values_equal(expected, actual)
            unit_evaluable = bool(fact["source"].get("unit_prediction_path"))
            unit_exact = normalize_unit(actual_unit, fact["expected"]["unit"]) == fact["expected"]["unit"] if unit_evaluable and presence == "available" else None
            atomic_exact = bool(value_exact and (unit_exact is not False))
            unsafe = presence == "available" and not atomic_exact
            details.append({
                "run_name": spec["run_name"],
                "model_group": spec["model_group"],
                "issuer": card["issuer"],
                "card_name": card["card_name"],
                "fact_id": fact["fact_id"],
                "benefit_or_fee_id": fact["benefit_or_fee_id"],
                "risk_component": fact["risk_component"],
                "page_num": fact["source"]["page_num"],
                "group_path": fact["relation"]["group_path"],
                "group_type": fact["relation"]["group_type"],
                "evaluation_role": fact["source"]["evaluation_role"],
                "prediction_supported": int(fact["source"]["prediction_supported"]),
                "presence": presence,
                "atomic_relation_exact": int(atomic_exact),
                "numeric_fact": int(isinstance(expected, (int, float)) and not isinstance(expected, bool)),
                "value_exact": int(value_exact),
                "unit_evaluable": int(unit_evaluable),
                "unit_exact": "" if unit_exact is None else int(unit_exact),
                "unsafe_mismatch": int(unsafe),
                "error_type": classify_error(fact, presence, value_exact, unit_exact),
                "expected_value": json.dumps(expected, ensure_ascii=False),
                "actual_value": json.dumps(actual, ensure_ascii=False),
                "expected_unit": fact["expected"]["unit"],
                "actual_unit": "" if actual_unit is None else str(actual_unit),
            })

write_csv(RUN_ROOT / "atomic_fact_details.csv", details)
print(f"원자 fact 상세 평가: {len(details)}행")

원자 fact 상세 평가: 2325행


In [4]:
relation_rows: list[dict[str, Any]] = []
relation_buckets: dict[tuple[str, ...], list[dict[str, Any]]] = defaultdict(list)
for row in details:
    if not row["prediction_supported"] or row["evaluation_role"] != "relation_primary":
        continue
    key = (
        row["run_name"], row["model_group"], row["issuer"], row["card_name"],
        row["benefit_or_fee_id"], row["group_path"], row["group_type"],
    )
    relation_buckets[key].append(row)

for key, rows in relation_buckets.items():
    run_name, model_group, issuer, card_name, benefit_id, group_path, group_type = key
    relation_rows.append({
        "run_name": run_name,
        "model_group": model_group,
        "issuer": issuer,
        "card_name": card_name,
        "benefit_or_fee_id": benefit_id,
        "group_path": group_path,
        "group_type": group_type,
        "atomic_facts": len(rows),
        "relation_group_exact": int(all(row["atomic_relation_exact"] for row in rows)),
        "unsafe_mismatches": sum(row["unsafe_mismatch"] for row in rows),
        "missing_predictions": sum(row["presence"] == "missing" for row in rows),
    })

write_csv(RUN_ROOT / "relation_group_details.csv", relation_rows)


def ratio(numerator: int, denominator: int) -> float | None:
    return round(numerator / denominator, 6) if denominator else None


run_summaries = []
for spec in SOURCE_SPECS:
    all_supported = [row for row in details if row["run_name"] == spec["run_name"] and row["prediction_supported"]]
    rows = [row for row in all_supported if row["evaluation_role"] == "relation_primary"]
    numeric_rows = [row for row in all_supported if row["numeric_fact"]]
    unit_rows = [row for row in all_supported if row["unit_evaluable"]]
    groups = [row for row in relation_rows if row["run_name"] == spec["run_name"]]
    table_rows = [row for row in groups if row["group_type"] == "table_row"]
    run_summaries.append({
        "run_name": spec["run_name"],
        "model_group": spec["model_group"],
        "supported_relation_facts": len(rows),
        "supported_numeric_probe_facts": len(numeric_rows),
        "atomic_relation_exact_rate": ratio(sum(row["atomic_relation_exact"] for row in rows), len(rows)),
        "numeric_value_accuracy": ratio(sum(row["value_exact"] for row in numeric_rows), len(numeric_rows)),
        "unit_accuracy": ratio(sum(int(row["unit_exact"] or 0) for row in unit_rows), len(unit_rows)),
        "relation_group_exact_rate": ratio(sum(row["relation_group_exact"] for row in groups), len(groups)),
        "table_row_relation_exact_rate": ratio(sum(row["relation_group_exact"] for row in table_rows), len(table_rows)),
        "unsafe_mismatch_rate": ratio(sum(row["unsafe_mismatch"] for row in rows), len(rows)),
        "unsafe_numeric_mismatch_rate": ratio(sum(row["unsafe_mismatch"] for row in numeric_rows), len(numeric_rows)),
        "missing_prediction_rate": ratio(sum(row["presence"] == "missing" for row in rows), len(rows)),
    })

write_csv(RUN_ROOT / "run_summary.csv", run_summaries)

metric_names = [
    "atomic_relation_exact_rate", "numeric_value_accuracy", "unit_accuracy",
    "relation_group_exact_rate", "table_row_relation_exact_rate",
    "unsafe_mismatch_rate", "unsafe_numeric_mismatch_rate", "missing_prediction_rate",
]
model_summaries = []
for model_group in dict.fromkeys(row["model_group"] for row in run_summaries):
    rows = [row for row in run_summaries if row["model_group"] == model_group]
    summary: dict[str, Any] = {"model_group": model_group, "runs": len(rows)}
    for metric in metric_names:
        values = [row[metric] for row in rows if row[metric] is not None]
        summary[f"{metric}_mean"] = round(statistics.mean(values), 6) if values else None
        summary[f"{metric}_population_stddev"] = round(statistics.pstdev(values), 6) if len(values) > 1 else None
        summary[f"{metric}_min"] = min(values) if values else None
        summary[f"{metric}_max"] = max(values) if values else None
    model_summaries.append(summary)

write_csv(RUN_ROOT / "model_repeatability_summary.csv", model_summaries)

error_rows = []
for (run_name, model_group), rows in {(row["run_name"], row["model_group"]): [] for row in details}.items():
    selected = [row for row in details if row["run_name"] == run_name and row["prediction_supported"]]
    counts = Counter(row["error_type"] for row in selected)
    for error_type, count in sorted(counts.items()):
        error_rows.append({
            "run_name": run_name, "model_group": model_group,
            "error_type": error_type, "count": count,
            "rate": ratio(count, len(selected)),
        })
write_csv(RUN_ROOT / "error_taxonomy.csv", error_rows)

result = {
    "schema_version": "relational_critical_fact_evaluation_v1",
    "run_id": RUN_ID,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "gold_path": str(V2_PATH.relative_to(PROJECT_ROOT)),
    "label_audit": audit_rows,
    "run_summary": run_summaries,
    "model_repeatability_summary": model_summaries,
    "limitations": [
        "관계 정확도는 09의 고정 JSON 스키마 경로에 배치된 값의 정확도다. 자유 형식 tuple 추출 정확도와 동일하지 않다.",
        "safety_audit_addition 라벨은 09 추출 요청에 없었으므로 이번 점수 분모에서 제외했다.",
        "Upstage는 1회 결과이므로 반복 분산을 계산할 수 없다.",
        "문자열은 Unicode NFKC, 대소문자, 공백만 정규화하며 의미 유사도를 임의 인정하지 않는다.",
    ],
}
write_json(RUN_ROOT / "summary.json", result)

OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
latest_pointer = {
    "latest_run_id": RUN_ID,
    "summary": str((RUN_ROOT / "summary.json").relative_to(PROJECT_ROOT)),
}
write_json(OUTPUT_ROOT / "latest_run.json", latest_pointer)

print(json.dumps(run_summaries, ensure_ascii=False, indent=2))
print(f"실행 결과: {RUN_ROOT.relative_to(PROJECT_ROOT)}")

[
  {
    "run_name": "upstage_baseline",
    "model_group": "Upstage Document Parse",
    "supported_relation_facts": 366,
    "supported_numeric_probe_facts": 270,
    "atomic_relation_exact_rate": 0.833333,
    "numeric_value_accuracy": 0.955556,
    "unit_accuracy": 0.916667,
    "relation_group_exact_rate": 0.767442,
    "table_row_relation_exact_rate": 0.766917,
    "unsafe_mismatch_rate": 0.117486,
    "unsafe_numeric_mismatch_rate": 0.033333,
    "missing_prediction_rate": 0.04918
  },
  {
    "run_name": "luna_original_repeat_1",
    "model_group": "OpenAI API Luna original",
    "supported_relation_facts": 366,
    "supported_numeric_probe_facts": 270,
    "atomic_relation_exact_rate": 0.860656,
    "numeric_value_accuracy": 0.996296,
    "unit_accuracy": 1.0,
    "relation_group_exact_rate": 0.776744,
    "table_row_relation_exact_rate": 0.774436,
    "unsafe_mismatch_rate": 0.139344,
    "unsafe_numeric_mismatch_rate": 0.003704,
    "missing_prediction_rate": 0.0
  },
  {
 

## 해석 기준

- **원자 관계 정확도**: 지정된 혜택/조건 경로에 놓인 한 값이 정확한 비율이다.
- **관계 그룹 정확도**: 한 혜택 또는 표의 한 행에 속한 대상·조건·값이 모두 맞아야 통과한다.
- **위험 오답률**: 값을 누락한 경우가 아니라, 존재하는 값을 틀리게 단정한 비율이다.
- **평가 미지원**: v2 안전성 감사에서 새로 추가했지만 09 구조화 프롬프트가 요청하지 않았던 항목이다. 다음 구조화 실험에서 반드시 포함한다.

이 노트북의 점수는 기존 OCR 모델을 최종 선정하기 위한 더 세밀한 근거지만, 고정 스키마가 관계 ID를 미리 제공한 평가라는 한계가 있다.
운영 전에는 v2 전체를 대상으로 자유 형식 관계 추출과 근거 페이지/블록 연결을 별도로 검증해야 한다.